# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets and their @id
print('Available record sets:')
for rs in metadata.record_sets:
    print(f"- Record Set Name: {rs.name}, @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field Name: {field.name}, @id: {field.id}, dataType: {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Select record sets by @id (update according to discovered @id in previous step)
# Example: record_set_ids = ['cr:ProteinTable', 'cr:EvidenceTable']
record_set_ids = [rs.id for rs in metadata.record_sets]  # collect all record set IDs
dataframes = {}

for record_set_id in record_set_ids:
    # Extract records for each record set
    print(f"Loading record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    if not dataframes[record_set_id].empty:
        print(f"Columns ({record_set_id}): {dataframes[record_set_id].columns.tolist()}")
        print(dataframes[record_set_id].head())
    else:
        print(f"No records loaded for {record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a record set with numeric data for EDA. Update these after inspecting columns above.
example_record_set_id = None
example_numeric_field = None
example_group_field = None

# Try to auto-detect one numeric field and one categorical/group field
for rs in metadata.record_sets:
    df = dataframes.get(rs.id)
    if df is not None and not df.empty:
        for field in rs.fields:
            # Check for numeric types
            if field.data_type in ["schema:Integer", "schema:Float", "schema:Number"] and field.id in df.columns:
                example_record_set_id = rs.id
                example_numeric_field = field.id
                # Try to select group field distinct from numeric field
                for f in rs.fields:
                    if f.id != field.id and f.data_type in ["schema:Text"] and f.id in df.columns:
                        example_group_field = f.id
                        break
                break
    if example_record_set_id is not None:
        break

if example_record_set_id is None or example_numeric_field is None:
    print("No suitable record set and numeric field found for EDA. Please check available data.")
else:
    df = dataframes[example_record_set_id]
    print(f"Using record set: {example_record_set_id}, numeric field: {example_numeric_field}, group field: {example_group_field}")
    # Filtering: use median as an example threshold
    threshold = df[example_numeric_field].median() if pd.api.types.is_numeric_dtype(df[example_numeric_field]) else 0
    filtered_df = df[df[example_numeric_field] > threshold]
    print(f"Filtered records with {example_numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{example_numeric_field}_normalized"] = (
        filtered_df[example_numeric_field] - filtered_df[example_numeric_field].mean()
    ) / filtered_df[example_numeric_field].std()
    print(f"Normalized {example_numeric_field} for filtered records:")
    print(filtered_df[[example_numeric_field, f"{example_numeric_field}_normalized"]].head())

    # Grouping
    if example_group_field is not None and example_group_field in df.columns:
        grouped_df = filtered_df.groupby(example_group_field)[example_numeric_field].mean().reset_index()
        print(f"Grouped data by {example_group_field} (mean {example_numeric_field}):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

if example_record_set_id and example_numeric_field:
    df = dataframes[example_record_set_id]
    plt.figure(figsize=(8, 4))
    df[example_numeric_field].hist(bins=30)
    plt.title(f'Distribution of {example_numeric_field} in {example_record_set_id}')
    plt.xlabel(example_numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    if example_group_field and example_group_field in df.columns:
        plt.figure(figsize=(10, 5))
        df.boxplot(column=example_numeric_field, by=example_group_field, rot=60)
        plt.title(f'{example_numeric_field} by {example_group_field}')
        plt.suptitle('')
        plt.xlabel(example_group_field)
        plt.ylabel(example_numeric_field)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

_In this notebook, we've loaded the FAIR² dataset package from Croissant using mlcroissant, automatically discovered available record sets, and performed sample exploratory analysis. For advanced analyses, consult dataset documentation and extend analysis as needed._